# Student Performance Predictor
### Training Notebook

This notebook walks through the full ML pipeline step-by-step: loading data, cleaning it, exploring it visually, training three models, comparing them, and saving the best one.

> This is the notebook version of `train.py`. You can also just run `python train.py` from the terminal to do all of this at once.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

sns.set_style("whitegrid")
%matplotlib inline

## 1. Load the Dataset

In [ ]:
df = pd.read_csv("../dataset/StudentsPerformance.csv")
print(df.shape)
df.head()

## 2. Data Preprocessing
### 2.1 Check missing values

In [ ]:
df.isnull().sum()

### 2.2 Fill missing numeric values with the median

In [ ]:
numeric_cols = ["math_score", "reading_score", "writing_score"]
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())
df.isnull().sum()

### 2.3 Remove duplicate rows

In [ ]:
print("Duplicates found:", df.duplicated().sum())
df = df.drop_duplicates()
df.shape

### 2.4 Encode categorical variables

In [ ]:
categorical_cols = ["gender", "race_ethnicity", "parental_level_of_education",
                    "lunch", "test_preparation_course"]

label_encoders = {}
df_encoded = df.copy()

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

df_encoded.head()

### 2.5 Feature Selection & Train/Test Split

In [ ]:
feature_columns = ["gender", "lunch", "test_preparation_course",
                    "reading_score", "writing_score"]
target_column = "math_score"

X = df_encoded[feature_columns]
y = df_encoded[target_column]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train size:", X_train.shape, "Test size:", X_test.shape)

### 2.6 Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Exploratory Data Analysis (EDA)

### 3.1 Distribution of Scores
Most scores form a roughly bell-shaped (normal) distribution centered around 65-70.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, numeric_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

### 3.2 Correlation Heatmap
Reading and writing scores are usually the most strongly correlated, since both are language-based skills.

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

### 3.3 Gender vs Performance

In [ ]:
df.groupby("gender")[numeric_cols].mean().plot(kind="bar", figsize=(7,5))
plt.title("Average Scores by Gender")
plt.xticks(rotation=0)
plt.show()

### 3.4 Lunch Type vs Performance
Standard lunch is generally associated with higher scores.

In [ ]:
df.groupby("lunch")[numeric_cols].mean().plot(kind="bar", figsize=(7,5))
plt.title("Average Scores by Lunch Type")
plt.xticks(rotation=0)
plt.show()

### 3.5 Test Preparation vs Performance
Completing the test prep course is linked to higher average scores.

In [ ]:
df.groupby("test_preparation_course")[numeric_cols].mean().plot(kind="bar", figsize=(7,5))
plt.title("Average Scores by Test Preparation")
plt.xticks(rotation=0)
plt.show()

### 3.6 Pair Plot

In [ ]:
sns.pairplot(df[numeric_cols + ["gender"]], hue="gender", diag_kind="kde")
plt.show()

### 3.7 Box Plots
Box plots reveal the spread and outliers of scores across categories.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sns.boxplot(data=df, x="test_preparation_course", y="math_score", ax=axes[0])
sns.boxplot(data=df, x="lunch", y="reading_score", ax=axes[1])
sns.boxplot(data=df, x="gender", y="writing_score", ax=axes[2])
plt.tight_layout()
plt.show()

## 4. Model Training
We train three simple, well-understood regression models.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=5, random_state=42),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42),
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model
    print("Trained:", name)

## 5. Model Evaluation

In [ ]:
results = []
for name, model in trained_models.items():
    y_pred = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    results.append({"Model": name, "MAE": round(mae,3), "MSE": round(mse,3),
                     "RMSE": round(rmse,3), "R2 Score": round(r2,3)})

results_df = pd.DataFrame(results).sort_values(by="R2 Score", ascending=False)
results_df

**Metric meanings:**
- **MAE**: average absolute error in score points (lower = better)
- **MSE**: average squared error, punishes big mistakes more (lower = better)
- **RMSE**: square root of MSE, back in score-point units (lower = better)
- **R² Score**: proportion of variance explained by the model (higher = better, max 1.0)

## 6. Save the Best Model

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print("Best model:", best_model_name)

joblib.dump(best_model, "../model/student_model.pkl")
joblib.dump(scaler, "../model/scaler.pkl")
joblib.dump(label_encoders, "../model/label_encoders.pkl")
joblib.dump(feature_columns, "../model/feature_columns.pkl")
joblib.dump(best_model_name, "../model/best_model_name.pkl")
print("Model saved successfully!")